In [ ]:
import sys, os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))
import numpy as np
from EDX import *
from utils import *
from utils_sofima import *
import hyperspy.api as hs
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import copy
from importlib import reload
from sklearn.preprocessing import MinMaxScaler

from skimage import data, img_as_float, feature
import tensorstore as ts
from scipy.stats import pearsonr
from datetime import datetime
import warnings
import pickle
from datetime import datetime
import pickle

from itertools import repeat
import random
import logging
import exspy

## Load aligned dataset

In [ ]:
# Unaligned
with open(f'/Volumes/Microscopy4/results/preprocessed_edx/20251201_142554_tile_aligned_20frames.pkl', 'rb') as file:    # I should re-align these later
    tile = pickle.load(file)

tile.apply("crop", parameters = {"crop_idx": (slice(50,tile.EDX_dim[0]-50),slice(50,tile.EDX_dim[1]-50),slice(None,None,None))})
tile.apply("binning", parameters={"dim": (974,974,250)})
tile.apply("MeanFilterEDX", parameters={"kernel_size": 3})

## Visualize/check dataset

In [ ]:
f, ax = plt.subplots(1,2, figsize = (10,5))
ax[0].imshow(tile.haadf, cmap='gray_r')
ax[0].set_title('HAADF', color='w')
ax[1].imshow(tile.FalseColor())
ax[1].set_title('NPS')
plt.close()

make_dark_presentation(f,text_color='white', line_width=2.5, transparent=True)

## Denoise with different values of Sigmna

In [ ]:
sigma_list = [0.1,0.2,0.3,0.4,0.5,0.6,0.7]

for cnt, sigma in enumerate(sigma_list):
    # start timer
    start = datetime.now()
    tile_denoised = tile.apply("NGMeet_matlab", parameters = {"sigma": sigma, "nEM": 10, "nIter": 2}, copy_instance=True)

    with open(f"/Volumes/Microscopy4/results/preprocessed_edx/aligned20_NGMeet_varying_Sigma/NGMeet_{sigma:.2f}.pkl", "wb") as f:
        pickle.dump(tile_denoised, f)

    end = datetime.now()
    duration = end - start
    duration_in_minutes_and_seconds = str(duration).split('.')[0][2:]

    print(f"Elapsed time for preprocessing {cnt}: {duration_in_minutes_and_seconds}")